In [25]:
# 1. Import required modules
# StateGraph, START, END: LangGraph core components for building workflows.
# TypedDict, NotRequired: For defining the state schema type hints.
# ChatGoogleGenerativeAI: LangChain wrapper for Google's Gemini LLM.
# load_dotenv: Loads environment variables (like GOOGLE_API_KEY) from a .env file.
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, NotRequired
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()


True

In [26]:
# 2. Initialize the LLM
# We create an instance of the Gemini 2.5 Flash model.
# temperature=0.7 allows for a slight bit of creativity in the response.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [27]:
# 3. Define the Graph State (Shared Memory)
# The state schema defines the data that flows through the graph.
# input: The user's question (required to start).
# output: The LLM's answer (NotRequired because it's generated by the node).
class SimpleLlmState(TypedDict):
    input: str
    output: NotRequired[str]

In [28]:
# 4. Define the Node (Worker/Action)
# This function takes the current state, formats a prompt, and calls the LLM.
# It returns a dict containing only the 'output' key, which LangGraph
# will automatically merge into the global state.
def qa_llm(state: SimpleLlmState) -> dict:
    prompt = f"Answer the following question : {state['input']}"
    response = llm.invoke(prompt)
    return {"output": response.content}

In [29]:
# 5. Build and Compile the Graph
# Initialize the graph with our SimpleLlmState schema.
graph = StateGraph(SimpleLlmState)

# Add the LLM node and define the sequential flow: START -> qa_llm -> END.
graph.add_node("qa_llm", qa_llm)
graph.add_edge(START, "qa_llm")
graph.add_edge("qa_llm", END)

# Finally, compile it into a runnable workflow object.
workflow = graph.compile()

In [30]:
# 6. Execute the workflow
# Invoke the graph with the initial state containing the user's question.
result = workflow.invoke({"input": "What is the capital of France?"})

# Extract and print the output string from the final state.
# We also clean up any markdown formatting (like ** for bold).
clean_output = result["output"].replace("**", "")
print(clean_output)

The capital of France is Paris.
